# All Phases Comparison
**CSC14120 - Parallel Programming**

Chạy cả Phase 2 và Phase 3, so sánh hiệu năng.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import files
import zipfile, os, urllib.request, tarfile

print("Upload file zip project:")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('project')

for root, dirs, _ in os.walk('project'):
    if 'src' in dirs:
        %cd {root}
        break

os.makedirs('data', exist_ok=True)
if not os.path.exists('data/data_batch_1.bin'):
    print('Downloading CIFAR-10...')
    urllib.request.urlretrieve('https://www.cs.toronto.edu/~kriz/cifar-10-binary.tar.gz', 'data/cifar.tar.gz')
    with tarfile.open('data/cifar.tar.gz', 'r:gz') as tar:
        tar.extractall('data')
    !mv data/cifar-10-batches-bin/* data/
print('Ready!')

In [ ]:
# Build cả 2 versions
!nvcc -O2 -std=c++17 -arch=sm_75 --expt-relaxed-constexpr -Iinclude \
    -o gpu_train src/main_gpu.cu src/layers_gpu.cu src/gpu_autoencoder.cu src/dataset.cpp

!nvcc -O2 -std=c++17 -arch=sm_75 --expt-relaxed-constexpr -DUSE_OPTIMIZED_KERNELS -Iinclude \
    -o gpu_train_opt src/main_gpu.cu src/layers_gpu.cu src/gpu_autoencoder.cu src/layers_gpu_opt.cu src/dataset.cpp
print('Build complete!')

In [ ]:
# Phase 2
print("="*50)
print("PHASE 2: GPU NAIVE")
print("="*50)
!./gpu_train --data data --epochs 10 --batch 64 --log p2.csv --log-txt p2.txt

In [ ]:
# Phase 3
print("="*50)
print("PHASE 3: GPU OPTIMIZED")
print("="*50)
!./gpu_train_opt --data data --epochs 10 --batch 64 --log p3.csv --log-txt p3.txt

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df2 = pd.read_csv('p2.csv'); ep2 = df2[df2['batch'].isna()]
df3 = pd.read_csv('p3.csv'); ep3 = df3[df3['batch'].isna()]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(ep2['epoch'], ep2['loss'], 'b-o', label='Phase 2')
axes[0].plot(ep3['epoch'], ep3['loss'], 'r-s', label='Phase 3')
axes[0].set_title('Training Loss'); axes[0].legend(); axes[0].grid(True)

# Time
axes[1].plot(ep2['epoch'], ep2['epoch_time_sec'], 'b-o', label='Phase 2')
axes[1].plot(ep3['epoch'], ep3['epoch_time_sec'], 'r-s', label='Phase 3')
axes[1].set_title('Epoch Time (s)'); axes[1].legend(); axes[1].grid(True)

# Speedup
speedup = ep2['epoch_time_sec'].values / ep3['epoch_time_sec'].values
axes[2].bar(ep3['epoch'], speedup, color='green')
axes[2].axhline(speedup.mean(), color='red', linestyle='--', label=f'Avg: {speedup.mean():.2f}x')
axes[2].set_title('Speedup (Phase3 vs Phase2)'); axes[2].legend(); axes[2].grid(True)

plt.tight_layout(); plt.savefig('comparison.png'); plt.show()

In [ ]:
# Summary
t2, t3 = ep2['epoch_time_sec'].mean(), ep3['epoch_time_sec'].mean()
print("="*60)
print("PERFORMANCE SUMMARY")
print("="*60)
print(f"{'Metric':<25} {'Phase 2':<15} {'Phase 3':<15} {'Speedup':<10}")
print("-"*65)
print(f"{'Avg Epoch Time (s)':<25} {t2:<15.2f} {t3:<15.2f} {t2/t3:<10.2f}x")
print(f"{'Total Time (s)':<25} {ep2['epoch_time_sec'].sum():<15.2f} {ep3['epoch_time_sec'].sum():<15.2f}")
print(f"{'Final Loss':<25} {ep2['loss'].iloc[-1]:<15.6f} {ep3['loss'].iloc[-1]:<15.6f}")

In [ ]:
files.download('comparison.png')
files.download('p2.csv')
files.download('p3.csv')